# shogi-human-policy を試す

棋譜ファイルやSFENから、人間が次に指す手の確率を予測します。
標準のCPUランタイムで、上から順に実行してください。

最初のセルで、v0.1.1のコードとモデルを自動取得します。
手元のモデルを使う場合は「手元ファイル」を選び、wheel、manifest.json、policy.safetensorsをアップロードしてください。


In [ ]:
# @title モデルの準備
import re
import subprocess
import sys
import urllib.request
from pathlib import Path

from google.colab import files

setup_mode = "リリース"  # @param ["リリース", "手元ファイル"]
release_version = "v0.1.1"  # @param {type:"string"}

model_directory = Path("model")
model_directory.mkdir(exist_ok=True)
if setup_mode == "リリース":
    if not re.fullmatch(r"v[0-9]+\.[0-9]+\.[0-9]+", release_version):
        raise ValueError("バージョンは v0.1.1 の形式で指定してください。")
    repository = "https://github.com/nyoki-mtl/shogi-human-policy"
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            f"git+{repository}.git@{release_version}",
        ]
    )
    for name in ("manifest.json", "policy.safetensors"):
        urllib.request.urlretrieve(
            f"{repository}/releases/download/{release_version}/{name}",
            model_directory / name,
        )
else:
    uploaded = files.upload()
    wheels = [name for name in uploaded if name.endswith(".whl")]
    if len(wheels) != 1 or not {"manifest.json", "policy.safetensors"}.issubset(uploaded):
        raise ValueError("wheelを1つと、manifest.json、policy.safetensorsを一緒に選んでください。")
    wheel = Path(Path(wheels[0]).name)
    wheel.write_bytes(uploaded[wheels[0]])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", str(wheel)])
    for name in ("manifest.json", "policy.safetensors"):
        (model_directory / name).write_bytes(uploaded[name])

In [ ]:
import torch
from IPython.display import SVG, Markdown, display

from shogi_human_policy import Conditions, PolicyModel, Position

torch.set_num_threads(4)
model = PolicyModel.from_directory(model_directory)
print(f"モデル読み込み完了：{model.manifest['model_version']} EMA")

## 局面を選ぶ

文字列には `startpos`、SFEN、`position startpos moves ...`、`SFEN moves ...` を入力できます。
棋譜アップロードでは `.kif` / `.kifu` / `.ki2` / `.csa` を選べます。
UTF-8とShift-JISに対応し、変化がある場合は本譜を使います。

In [ ]:
# @title 入力方法
input_mode = "文字列"  # @param ["文字列", "棋譜ファイル"]
position_text = "startpos moves 7g7f 3c3d"  # @param {type:"string"}

if input_mode == "文字列":
    full_position = Position.from_usi(position_text)
else:
    record_upload = files.upload()
    if len(record_upload) != 1:
        raise ValueError("棋譜ファイルを1つ選んでください。")
    record_name, record_bytes = next(iter(record_upload.items()))
    record_path = Path(Path(record_name).name)
    record_path.write_bytes(record_bytes)
    full_position = Position.from_file(record_path)
print(f"開始局面から{len(full_position.moves)}手の着手列を読み込みました。")

In [ ]:
# @title 何手進めた局面を使うか
# @markdown 空欄は末尾、0は開始局面、30は開始局面から30手進めた位置です。
moves_played = ""  # @param {type:"string"}

selected_ply = None if not moves_played.strip() else int(moves_played)
position = full_position.at_ply(selected_ply)
board = position.to_board()
side_name = "先手" if int(board.turn) == 0 else "後手"
display(Markdown(f"**{side_name}の手番** ／ 次は{board.to_sfen().split()[3]}手目"))
display(SVG(str(board.to_svg())))
print(position.to_usi())

## 条件を選んで予測

Rは将棋倶楽部24の尺度です。先手と後手の値を入力すると、現在の手番に合わせて自分と相手へ振り分けます。
駒落ちは先手欄を下手、後手欄を上手として入力します。
時計は**選んだ局面で、着手前に使える秒数**です。本時間と秒読みがあれば合算します。
棋譜ヘッダや当該手の消費時間からは推測しません。不明な条件はチェックを外してください。

値を変えてこのセルを再実行すると、同じ局面で条件による変化を確認できます。

In [ ]:
# @title レーティングと時計
use_ratings = True  # @param {type:"boolean"}
black_rating = 1500  # @param {type:"number"}
white_rating = 1800  # @param {type:"number"}
use_clocks = False  # @param {type:"boolean"}
black_seconds = 60  # @param {type:"number"}
white_seconds = 60  # @param {type:"number"}
top_k = 5  # @param {type:"integer"}


def current_pair(enabled, black, white):
    if not enabled:
        return None
    return (black, white) if int(board.turn) == 0 else (white, black)


conditions = Conditions(
    ratings=current_pair(use_ratings, black_rating, white_rating),
    clocks=current_pair(use_clocks, black_seconds, white_seconds),
)
result = model.predict(position, conditions, top_k=top_k)
rows = result.as_rows(position)
if not rows:
    display(Markdown("この局面に合法手はありません。"))
else:
    table = ["| 順位 | 指し手 | USI | 確率 |", "|---:|---|---|---:|"]
    for rank, row in enumerate(rows, 1):
        table.append(f"| {rank} | {row['notation']} | `{row['usi']}` | {row['probability']:.2%} |")
    display(Markdown("\n".join(table)))

候補の確率は全合法手を分母にしています。
履歴付きの予測には、棋譜ファイルや着手列を入力してください。